# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. 

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print("Dataset Name:", meta.name)
print("Description:", meta.description)

## 2. Data Overview
List available `RecordSet` objects and their `@id`s. Then, list fields for each record set.

In [ ]:
# List all Record Sets in the dataset
record_sets = dataset.metadata.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"  RecordSet name: {rs.name}, @id: {rs['@id']}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      {field.name} (@id: {field['@id']}), type: {getattr(field,'data_type',None)}")
    print()

## 3. Data Extraction
Load data from a selected `RecordSet` using its `@id` and display its columns. Use the actual `@id` field when referring to each item.

In [ ]:
# For demonstration, let's pick the first RecordSet
record_set_ids = [rs['@id'] for rs in record_sets]

print("Extracting data from these Record Sets:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records):
        dataframes[record_set_id] = pd.DataFrame(records)

# Examine the columns of the first non-empty dataframe
for rsid, df in dataframes.items():
    print(f"First rows of Record Set @id: {rsid}")
    print("Columns:", df.columns.tolist())
    display(df.head())
    break  # only display one for brevity

## 4. Exploratory Data Analysis (EDA)
Let's filter records, normalize a numeric field, and group by a categorical field, referencing all fields by their `@id`.

In [ ]:
# For demo, automatically detect a numeric field and a group field
first_record_set_id = list(dataframes.keys())[0]
df = dataframes[first_record_set_id]

# Get field info for this record set
fields = None
for rs in record_sets:
    if rs['@id'] == first_record_set_id:
        fields = rs.fields
        break
# Choose a numeric field
numeric_field_id = None
group_field_id = None
for field in fields:
    # Pick first field identified as Float/Number/Integer
    if getattr(field, 'data_type', None) in ['Float', 'Number', 'Integer'] and field['@id'] in df.columns:
        numeric_field_id = field['@id']
        break
# Pick a group field (Text/categorical)
for field in fields:
    if getattr(field, 'data_type', None) in ['Text', None, 'String'] and field['@id'] in df.columns:
        group_field_id = field['@id']
        break

print(f"Numeric field selected: {numeric_field_id}")
print(f"Categorical/grouping field selected: {group_field_id}")

if numeric_field_id is not None and numeric_field_id in df.columns:
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field and grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        plt.figure(figsize=(10,5))
        group_means.nlargest(20).plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} grouped by {group_field_id} (Top 20)")
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading a Croissant FAIR² dataset, previewing its contents, filtering and normalizing numeric fields, and visualizing distributions. 

**Key Observations:**
- Record set and field discovery is enabled by `mlcroissant`'s metadata structure.
- Dynamic analysis and visualization pipelines can be composed for new datasets with minimal changes by referencing all fields by their `@id`.
- For in-depth analysis, refer to dataset documentation and field-level descriptions.